# Iris Segmentation U-Net (eye-crop based, full-resolution output)

**Pipeline this notebook implements:**

```
full-res face image
   -> eye corner points (from YOUR existing corner-detection model, or CVAT GT while training)
   -> crop a square region around each eye (at full resolution)
   -> resize crop to a fixed size and feed to U-Net
   -> U-Net predicts iris mask -> fit a circle to it
   -> map circle back through the crop transform -> center/radius in ORIGINAL full-image pixel coordinates
```

This only trains **one U-Net**, for iris segmentation. Corner/white-point detection stays in your other model — here we just *use* the corner points (from CVAT ground truth during training, or from your corner model at inference time) to know where to crop.

Two training samples come out of every labeled image (left eye crop + right eye crop), so your 120 labeled images give ~240 training crops.


In [ ]:
# ============ CONFIG ============
import os

CONFIG = {
    "annotations_root": "data/landmarks",   # recursively searched for *.xml (CVAT exports)
    "images_root": "data",                  # recursively searched for the actual jpg/png files
    "work_dir": "work",

    "iris_labels": {"left": "left_iris_mask", "right": "right_iris_mask"},
    # corner/white-point labels used ONLY to build a sensible crop box per eye (not trained on)
    "corner_labels": {
        "left":  ["left_eye_inner_corner", "left_eye_outer_corner", "left_white_point"],
        "right": ["right_eye_inner_corner", "right_eye_outer_corner", "right_white_point"],
    },

    "crop_margin": 1.8,     # crop box = corner-bbox expanded by this factor (>1 gives context around the eye)
    "crop_size": 256,       # resize each eye crop to this size for the U-Net

    "batch_size": 8,
    "epochs_frozen": 15,
    "epochs_finetune": 40,
    "lr_frozen": 1e-3,
    "lr_finetune": 1e-4,
    "val_fraction": 0.2,
    "seed": 42,
    "encoder": "resnet18",
    "device": "cuda",
}

for sub in ("masks", "checkpoints", "predictions", "crops"):
    os.makedirs(os.path.join(CONFIG["work_dir"], sub), exist_ok=True)
print(CONFIG)


## 0. Install dependencies (run once)

In [ ]:
# !pip install torch torchvision segmentation-models-pytorch albumentations opencv-python-headless scikit-learn matplotlib --quiet


In [ ]:
import numpy as np, cv2, torch, random, glob
import matplotlib.pyplot as plt
from pathlib import Path
import xml.etree.ElementTree as ET

random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
device = torch.device(CONFIG["device"] if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 1. Locate every annotation XML and every image file

Your annotations are split across many per-patient folders. This walks `annotations_root` for all `*.xml` files, and separately indexes every image under `images_root` by filename (so it doesn't matter which subfolder an image physically lives in, as long as filenames are unique — which they are here, e.g. `P003_IMG_8211.JPG`).

**If your images live somewhere outside `data/`, update `images_root`.**


In [ ]:
def find_xml_files(root):
    return sorted(glob.glob(os.path.join(root, "**", "*.xml"), recursive=True))

def index_images(root, exts=(".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG")):
    index = {}
    for ext in exts:
        for p in glob.glob(os.path.join(root, "**", f"*{ext}"), recursive=True):
            index[os.path.basename(p)] = p
    return index

xml_files = find_xml_files(CONFIG["annotations_root"])
image_index = index_images(CONFIG["images_root"])
print(f"Found {len(xml_files)} annotation XML files")
print(f"Indexed {len(image_index)} image files")


## 2. Parse all XMLs -> per-image iris polygons + corner points

For each `<image>` tag we pull out:
- `left_iris_mask` / `right_iris_mask` polygons (8 pts each) -> used as the segmentation target
- the corner/white-point labels for that side -> used only to build the crop box

Images missing a given side's iris polygon are simply skipped for that side (e.g. only one eye visible/labeled).


In [ ]:
def parse_points_str(points_str):
    pts = []
    for pair in points_str.strip().split(';'):
        x_str, y_str = pair.split(',')
        pts.append((float(x_str), float(y_str)))
    return np.array(pts, dtype=np.float32)

def fit_circle(points):
    x, y = points[:, 0], points[:, 1]
    A = np.column_stack([x, y, np.ones_like(x)])
    b = x**2 + y**2
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = sol[0] / 2.0, sol[1] / 2.0
    r = np.sqrt(sol[2] + cx**2 + cy**2)
    return float(cx), float(cy), float(r)

def parse_all_xml(xml_files, iris_labels, corner_labels):
    """Returns dict: filename -> {
        'width', 'height',
        'left':  {'iris_pts':..., 'cx','cy','r', 'corner_pts':...}  (if present)
        'right': {...} (if present)
    }"""
    records = {}
    for xml_path in xml_files:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        for image_tag in root.findall("image"):
            filename = image_tag.get("name")
            width = float(image_tag.get("width"))
            height = float(image_tag.get("height"))

            rec = records.setdefault(filename, {"width": width, "height": height})

            # gather all point-like shapes by label, regardless of tag (points/polygon/polyline)
            shapes_by_label = {}
            for tag_name in ("points", "polygon", "polyline"):
                for shape_tag in image_tag.findall(tag_name):
                    shapes_by_label.setdefault(shape_tag.get("label"), []).append(shape_tag)

            for side, iris_label in iris_labels.items():
                if iris_label not in shapes_by_label:
                    continue
                iris_pts = parse_points_str(shapes_by_label[iris_label][0].get("points"))
                if len(iris_pts) < 5:
                    continue
                cx, cy, r = fit_circle(iris_pts)

                corner_pts = []
                for label in corner_labels[side]:
                    if label in shapes_by_label:
                        corner_pts.append(parse_points_str(shapes_by_label[label][0].get("points"))[0])
                corner_pts = np.array(corner_pts, dtype=np.float32) if corner_pts else None

                rec[side] = {
                    "iris_pts": iris_pts, "cx": cx, "cy": cy, "r": r,
                    "corner_pts": corner_pts,
                }
    return records

records = parse_all_xml(xml_files, CONFIG["iris_labels"], CONFIG["corner_labels"])
n_left = sum(1 for r in records.values() if "left" in r)
n_right = sum(1 for r in records.values() if "right" in r)
print(f"Images with parsed annotations: {len(records)}  (left iris: {n_left}, right iris: {n_right})")


## 3. Build the crop box for each eye

Crop box = bounding box of the corner points (or, if corner points weren't labeled for that eye, the iris polygon's bounding box as a fallback), expanded by `crop_margin`, forced square, clipped to image bounds.

This same function is what you'll call at inference time using YOUR corner model's output — see Section 9.


In [ ]:
def points_bbox(points):
    x0, y0 = points[:, 0].min(), points[:, 1].min()
    x1, y1 = points[:, 0].max(), points[:, 1].max()
    return x0, y0, x1, y1

def square_crop_box(x0, y0, x1, y1, margin, img_w, img_h):
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    half_side = max(x1 - x0, y1 - y0) * margin / 2
    nx0, ny0 = cx - half_side, cy - half_side
    nx1, ny1 = cx + half_side, cy + half_side
    # clip to image bounds, preserving square-ness as best as possible
    nx0 = max(0, nx0); ny0 = max(0, ny0)
    nx1 = min(img_w, nx1); ny1 = min(img_h, ny1)
    return nx0, ny0, nx1, ny1

def get_crop_box(anchor_points, iris_pts_fallback, margin, img_w, img_h):
    pts = anchor_points if anchor_points is not None and len(anchor_points) >= 2 else iris_pts_fallback
    x0, y0, x1, y1 = points_bbox(pts)
    return square_crop_box(x0, y0, x1, y1, margin, img_w, img_h)


## 4. Build the (crop, mask, transform) dataset

For every (image, side) pair with a labeled iris:
1. Load the **full-resolution** image.
2. Compute the crop box, crop at full resolution.
3. Rasterize the iris polygon as a mask, in the *same original coordinates*, then crop it identically.
4. Resize crop + mask together to `crop_size`.
5. Save both to disk, plus the transform needed to map predictions back to full-image coordinates.


In [ ]:
from dataclasses import dataclass

@dataclass
class CropTransform:
    x0: float; y0: float; x1: float; y1: float  # crop box in ORIGINAL image coords
    resized_to: int

    def crop_to_full(self, cx, cy, r):
        scale = (self.x1 - self.x0) / self.resized_to
        full_cx = self.x0 + cx * scale
        full_cy = self.y0 + cy * scale
        full_r = r * scale
        return full_cx, full_cy, full_r

    def full_to_crop_point(self, x, y):
        scale = self.resized_to / (self.x1 - self.x0)
        return (x - self.x0) * scale, (y - self.y0) * scale


def build_eye_sample(filename, side, rec, image_index, crop_margin, crop_size):
    if filename not in image_index:
        return None
    img_path = image_index[filename]
    img = cv2.imread(img_path)  # full resolution, full quality
    if img is None:
        return None
    h, w = img.shape[:2]

    side_rec = rec[side]
    x0, y0, x1, y1 = get_crop_box(side_rec["corner_pts"], side_rec["iris_pts"], crop_margin, w, h)
    x0i, y0i, x1i, y1i = int(round(x0)), int(round(y0)), int(round(x1)), int(round(y1))
    if x1i - x0i < 10 or y1i - y0i < 10:
        return None

    crop = img[y0i:y1i, x0i:x1i]

    full_mask = np.zeros((h, w), dtype=np.uint8)
    poly = side_rec["iris_pts"].astype(np.int32)
    cv2.fillPoly(full_mask, [poly], 1)
    mask_crop = full_mask[y0i:y1i, x0i:x1i]

    crop_resized = cv2.resize(crop, (crop_size, crop_size), interpolation=cv2.INTER_AREA)
    mask_resized = cv2.resize(mask_crop, (crop_size, crop_size), interpolation=cv2.INTER_NEAREST)

    transform = CropTransform(x0i, y0i, x1i, y1i, crop_size)
    return crop_resized, mask_resized, transform


sample_ids = []   # list of (filename, side)
transforms = {}   # (filename, side) -> CropTransform
crops_dir = os.path.join(CONFIG["work_dir"], "crops")
masks_dir = os.path.join(CONFIG["work_dir"], "masks")

for filename, rec in records.items():
    for side in ("left", "right"):
        if side not in rec:
            continue
        result = build_eye_sample(filename, side, rec, image_index, CONFIG["crop_margin"], CONFIG["crop_size"])
        if result is None:
            continue
        crop_img, mask_img, transform = result
        sample_id = f"{Path(filename).stem}__{side}"
        cv2.imwrite(os.path.join(crops_dir, sample_id + ".png"), crop_img)
        cv2.imwrite(os.path.join(masks_dir, sample_id + ".png"), mask_img * 255)
        transforms[sample_id] = transform
        sample_ids.append(sample_id)

print(f"Built {len(sample_ids)} eye-crop samples from {len(records)} images")


## 5. Sanity-check a few crops + masks + circle fits

In [ ]:
def show_sample(sample_id, crops_dir, masks_dir):
    img = cv2.cvtColor(cv2.imread(os.path.join(crops_dir, sample_id + ".png")), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(os.path.join(masks_dir, sample_id + ".png"), cv2.IMREAD_GRAYSCALE)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img); axes[0].set_title(sample_id)
    axes[1].imshow(img); axes[1].imshow(mask, alpha=0.4, cmap="Reds"); axes[1].set_title("mask overlay")
    plt.show()

for sid in random.sample(sample_ids, min(4, len(sample_ids))):
    show_sample(sid, crops_dir, masks_dir)


## 6. Dataset & augmentations

Same reasoning as before: avoid elastic/grid distortion (warps the circle), keep flips/rotation/brightness/blur/noise.


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

IMG_SIZE = CONFIG["crop_size"]

train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.GaussNoise(p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(),
    ToTensorV2(),
])
val_tf = A.Compose([A.Normalize(), ToTensorV2()])

class IrisCropDataset(Dataset):
    def __init__(self, ids, crops_dir, masks_dir, transform):
        self.ids = ids; self.crops_dir = crops_dir; self.masks_dir = masks_dir; self.transform = transform
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        sid = self.ids[idx]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.crops_dir, sid + ".png")), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(os.path.join(self.masks_dir, sid + ".png"), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        aug = self.transform(image=img, mask=mask)
        return aug["image"], aug["mask"].unsqueeze(0).float(), sid

# split by PATIENT (filename prefix), not by crop, so left/right eyes of the
# same image never straddle train/val and inflate the val score
patient_ids = sorted(set(sid.split("__")[0] for sid in sample_ids))
train_patients, val_patients = train_test_split(patient_ids, test_size=CONFIG["val_fraction"], random_state=CONFIG["seed"])
train_ids = [s for s in sample_ids if s.split("__")[0] in train_patients]
val_ids = [s for s in sample_ids if s.split("__")[0] in val_patients]
print(f"Train crops: {len(train_ids)}  Val crops: {len(val_ids)}")

train_ds = IrisCropDataset(train_ids, crops_dir, masks_dir, train_tf)
val_ds = IrisCropDataset(val_ids, crops_dir, masks_dir, val_tf)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2)


## 7. Model: U-Net with pretrained encoder (frozen -> finetune)

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(encoder_name=CONFIG["encoder"], encoder_weights="imagenet", in_channels=3, classes=1, activation=None).to(device)

def set_encoder_trainable(model, trainable: bool):
    for p in model.encoder.parameters():
        p.requires_grad = trainable

print("Model ready:", CONFIG["encoder"])


In [ ]:
dice_loss_fn = smp.losses.DiceLoss(mode="binary", from_logits=True)
bce_loss_fn = torch.nn.BCEWithLogitsLoss()
def combined_loss(logits, targets):
    return dice_loss_fn(logits, targets) + bce_loss_fn(logits, targets)

@torch.no_grad()
def dice_score(logits, targets, eps=1e-7):
    preds = (torch.sigmoid(logits) > 0.5).float()
    inter = (preds * targets).sum(dim=(1,2,3))
    union = preds.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
    return ((2*inter+eps)/(union+eps)).mean().item()


In [ ]:
def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_dice, n = 0.0, 0.0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, masks, _ in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            loss = combined_loss(logits, masks)
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item(); total_dice += dice_score(logits, masks); n += 1
    return total_loss/n, total_dice/n

def train_stage(model, train_loader, val_loader, epochs, lr, stage_name, ckpt_path):
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)
    best_val = float("inf")
    hist = {"train_loss": [], "val_loss": [], "train_dice": [], "val_dice": []}
    for epoch in range(1, epochs+1):
        tl, td = run_epoch(train_loader, model, optimizer)
        vl, vd = run_epoch(val_loader, model, None)
        scheduler.step(vl)
        hist["train_loss"].append(tl); hist["val_loss"].append(vl)
        hist["train_dice"].append(td); hist["val_dice"].append(vd)
        if vl < best_val:
            best_val = vl
            torch.save(model.state_dict(), ckpt_path)
        print(f"[{stage_name}] {epoch}/{epochs} train_loss={tl:.4f} val_loss={vl:.4f} train_dice={td:.4f} val_dice={vd:.4f}")
    return hist


In [ ]:
set_encoder_trainable(model, False)
ckpt_frozen = os.path.join(CONFIG["work_dir"], "checkpoints", "stage1_frozen.pt")
history_frozen = train_stage(model, train_loader, val_loader, CONFIG["epochs_frozen"], CONFIG["lr_frozen"], "frozen", ckpt_frozen)


In [ ]:
model.load_state_dict(torch.load(ckpt_frozen, map_location=device))
set_encoder_trainable(model, True)
ckpt_final = os.path.join(CONFIG["work_dir"], "checkpoints", "stage2_finetuned.pt")
history_finetune = train_stage(model, train_loader, val_loader, CONFIG["epochs_finetune"], CONFIG["lr_finetune"], "finetune", ckpt_final)
model.load_state_dict(torch.load(ckpt_final, map_location=device))
model.eval()
print("Loaded best finetuned model.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tl = history_frozen["train_loss"] + history_finetune["train_loss"]
vl = history_frozen["val_loss"] + history_finetune["val_loss"]
td = history_frozen["train_dice"] + history_finetune["train_dice"]
vd = history_frozen["val_dice"] + history_finetune["val_dice"]
axes[0].plot(tl, label="train"); axes[0].plot(vl, label="val")
axes[0].axvline(CONFIG["epochs_frozen"], color="gray", ls="--"); axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(td, label="train"); axes[1].plot(vd, label="val")
axes[1].axvline(CONFIG["epochs_frozen"], color="gray", ls="--"); axes[1].set_title("Dice"); axes[1].legend()
plt.show()


## 8. Mask -> circle, mapped back to FULL-IMAGE coordinates

This is the key step for "full quality": the circle is fit in the resized-crop's pixel space, then `CropTransform.crop_to_full` maps it back to the original 3024x4032 image's coordinate system.


In [ ]:
def mask_to_circle(prob_mask, threshold=0.5):
    binary = (prob_mask > threshold).astype(np.uint8)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    biggest = max(contours, key=cv2.contourArea)
    pts = biggest.reshape(-1, 2).astype(np.float32)
    if len(pts) < 5:
        (cx, cy), r = cv2.minEnclosingCircle(biggest)
        return float(cx), float(cy), float(r)
    return fit_circle(pts)

@torch.no_grad()
def predict_circle_in_crop(model, crop_img_bgr):
    """crop_img_bgr: already-cropped eye region (any resolution). Returns circle in crop-resized coords, or None."""
    crop_resized = cv2.resize(crop_img_bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(crop_resized, cv2.COLOR_BGR2RGB)
    tensor = val_tf(image=rgb)["image"].unsqueeze(0).to(device)
    prob = torch.sigmoid(model(tensor))[0, 0].cpu().numpy()
    return mask_to_circle(prob)

def predict_iris_full_image(model, full_img_bgr, anchor_points, crop_margin=CONFIG["crop_margin"]):
    """anchor_points: Nx2 array of eye-corner points IN FULL-IMAGE COORDS (from your corner model, or CVAT GT).
    Returns (cx, cy, r) in full-image pixel coordinates, or None."""
    h, w = full_img_bgr.shape[:2]
    x0, y0, x1, y1 = points_bbox(anchor_points)
    x0, y0, x1, y1 = square_crop_box(x0, y0, x1, y1, crop_margin, w, h)
    x0i, y0i, x1i, y1i = int(round(x0)), int(round(y0)), int(round(x1)), int(round(y1))
    crop = full_img_bgr[y0i:y1i, x0i:x1i]
    if crop.size == 0:
        return None
    result = predict_circle_in_crop(model, crop)
    if result is None:
        return None
    cx, cy, r = result
    transform = CropTransform(x0i, y0i, x1i, y1i, IMG_SIZE)
    return transform.crop_to_full(cx, cy, r)


## 9. Evaluation: center/radius error in FULL-IMAGE pixels

This compares predictions to ground truth in the original 3024x4032 coordinate system (not crop-resized pixels), since that's the "full quality" accuracy that actually matters.


In [ ]:
def evaluate_full_image(model, val_ids, records, image_index):
    center_errors, radius_errors = [], []
    for sid in val_ids:
        filename_stem, side = sid.split("__")
        # recover original filename (may have any extension) via records lookup
        matches = [f for f in records if Path(f).stem == filename_stem]
        if not matches:
            continue
        filename = matches[0]
        rec = records[filename]
        if side not in rec:
            continue
        side_rec = rec[side]
        anchor = side_rec["corner_pts"] if side_rec["corner_pts"] is not None else side_rec["iris_pts"]

        img_path = image_index.get(filename)
        if img_path is None:
            continue
        full_img = cv2.imread(img_path)
        pred = predict_iris_full_image(model, full_img, anchor)
        if pred is None:
            continue
        pcx, pcy, pr = pred
        gcx, gcy, gr = side_rec["cx"], side_rec["cy"], side_rec["r"]
        center_errors.append(np.hypot(pcx-gcx, pcy-gcy))
        radius_errors.append(abs(pr-gr))

    print(f"N evaluated: {len(center_errors)}")
    print(f"Center error (full-image px): mean={np.mean(center_errors):.2f} median={np.median(center_errors):.2f} max={np.max(center_errors):.2f}")
    print(f"Radius error (full-image px): mean={np.mean(radius_errors):.2f} median={np.median(radius_errors):.2f} max={np.max(radius_errors):.2f}")
    return center_errors, radius_errors

_ = evaluate_full_image(model, val_ids, records, image_index)


## 10. Visualize a few predictions on the full-resolution image

In [ ]:
def show_full_prediction(sid, records, image_index):
    filename_stem, side = sid.split("__")
    matches = [f for f in records if Path(f).stem == filename_stem]
    if not matches: return
    filename = matches[0]
    rec = records[filename]
    side_rec = rec[side]
    anchor = side_rec["corner_pts"] if side_rec["corner_pts"] is not None else side_rec["iris_pts"]

    full_img = cv2.imread(image_index[filename])
    pred = predict_iris_full_image(model, full_img, anchor)

    x0, y0, x1, y1 = square_crop_box(*points_bbox(anchor), CONFIG["crop_margin"], full_img.shape[1], full_img.shape[0])
    crop = full_img[int(y0):int(y1), int(x0):int(x1)]
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(figsize=(5,5))
    ax.imshow(crop_rgb)
    if pred is not None:
        pcx, pcy, pr = pred
        lcx, lcy = pcx - x0, pcy - y0  # back into this crop's display coords
        ax.add_patch(plt.Circle((lcx, lcy), pr, color="cyan", fill=False, lw=2, label="predicted"))
    gcx, gcy = side_rec["cx"] - x0, side_rec["cy"] - y0
    ax.add_patch(plt.Circle((gcx, gcy), side_rec["r"], color="lime", fill=False, lw=2, ls="--", label="ground truth"))
    ax.legend(); ax.set_title(sid)
    plt.show()

for sid in random.sample(val_ids, min(4, len(val_ids))):
    show_full_prediction(sid, records, image_index)


## 11. Applying this to your remaining unlabeled images

Since iris cropping depends on knowing roughly where the eye is, you need eye-corner points for each new image before calling the iris model — that's exactly what your **existing corner-detection model** is for. The integration point is `predict_iris_full_image(model, full_img_bgr, anchor_points)`, where `anchor_points` is whatever your corner model outputs for one eye (as an array of full-image `(x, y)` points, 2+ points is enough — inner corner, outer corner, white point, etc.).

Example workflow once you have corner predictions saved (e.g. as a JSON `{filename: {"left": [[x,y],...], "right": [[x,y],...]}}`):


In [ ]:
import json

def run_bootstrap_inference(model, corner_predictions_path, image_index, out_xml_path,
                             iris_labels=CONFIG["iris_labels"]):
    with open(corner_predictions_path) as f:
        corner_preds = json.load(f)

    root = ET.Element("annotations")
    img_id = 0
    for filename, sides in corner_preds.items():
        if filename not in image_index:
            continue
        full_img = cv2.imread(image_index[filename])
        h, w = full_img.shape[:2]
        image_el = None

        for side, label in iris_labels.items():
            if side not in sides or len(sides[side]) < 2:
                continue
            anchor = np.array(sides[side], dtype=np.float32)
            pred = predict_iris_full_image(model, full_img, anchor)
            if pred is None:
                continue
            cx, cy, r = pred

            if image_el is None:
                image_el = ET.SubElement(root, "image", id=str(img_id), name=filename, width=str(w), height=str(h))
                img_id += 1

            angles = np.linspace(0, 2*np.pi, 8, endpoint=False)
            pts = [(cx + r*np.cos(a), cy + r*np.sin(a)) for a in angles]
            pts_str = ";".join(f"{x:.2f},{y:.2f}" for x, y in pts)
            ET.SubElement(image_el, "polygon", label=label, points=pts_str, occluded="0", source="auto")

    ET.ElementTree(root).write(out_xml_path, encoding="utf-8", xml_declaration=True)
    print(f"Wrote pre-annotations to {out_xml_path}")

# Example (uncomment once you have a corner-predictions JSON from your other model):
# run_bootstrap_inference(model, "work/corner_predictions.json", image_index, "work/iris_preannotations.xml")


### Notes / things to double check

- **`images_root`**: I recursively index every image under `data/` by filename. If your images live outside that tree, update `images_root` in the config cell.
- **`crop_margin` (1.8)**: controls how much context around the eye corners gets included. If irises are getting clipped at the crop edge in the sanity-check plots (Section 5/10), increase it; if the eye region is mostly empty space, decrease it.
- **Patient-level split**: train/val is split by patient ID prefix (`P003` etc.), not by crop, so both eyes of the same person never leak across the split.
- Once you've labeled more images, just re-run from Section 1 — everything downstream re-parses automatically.
